# 08a - Foundry Setup & Configuration

This notebook handles **infrastructure setup** and **configuration** for Azure AI Foundry:

- Load configuration from `foundry-config.json`
- Set up credentials (user or agent identity)
- Verify search connections
- Validate agent identity configuration

**Prerequisites:**
- Notebook 05 (Azure infrastructure deployed)
- Notebook 07 (Knowledge bases created)
- `foundry-config.json` exists

**Next:** Run `08b-foundry-agents.ipynb` for agent creation and queries.

In [40]:
# Install the agent framework packages if needed
# !pip install agent-framework-azure-ai --pre
# !pip install azure-identity

# =============================================================================
# IMPORTANT: Authentication for Agent Framework vs Agent Identities
# =============================================================================
# 
# This notebook uses the Microsoft Agent Framework to create chat agents.
# The framework requires authentication to Azure AI Foundry services.
#
# ⚠️  COMMON ERROR: AADSTS82001
# If you see: "Agentic application '...' is not permitted to request app-only 
# tokens for resource '...'"
#
# This happens when DefaultAzureCredential picks up an Agent Identity context
# (from environment variables or managed identity). Agent Identities use a 
# DIFFERENT token flow and cannot request app-only tokens directly.
#
# SOLUTIONS:
# 1. Use InteractiveBrowserCredential (this notebook's approach)
# 2. Use AzureCliCredential if you're logged in via `az login`
# 3. Clear any AZURE_CLIENT_ID env vars that point to agent identities
# 4. For production: Use Microsoft Entra SDK for Agent ID container
#
# See: https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/microsoft-entra-sdk-for-agent-identities
# =============================================================================

In [41]:
import os
import sys
import yaml
import warnings
import logging
from pathlib import Path
from dotenv import load_dotenv

# =============================================================================
# Suppress noisy warnings BEFORE importing Azure/aiohttp libraries
# =============================================================================
# These warnings are cosmetic and don't affect functionality
warnings.filterwarnings("ignore", message=".*Unclosed.*")
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", message=".*is not a known attribute.*")

# Suppress Azure SDK logging noise
logging.getLogger("azure").setLevel(logging.ERROR)
logging.getLogger("aiohttp").setLevel(logging.ERROR)

# Redirect stderr for aiohttp cleanup messages (they're printed directly, not through warnings)
class StderrFilter:
    """Filter stderr to suppress aiohttp/asyncio noise."""
    def __init__(self, stream):
        self.stream = stream
        self.suppress_patterns = [
            'Unclosed client session',
            'Unclosed connector',
            'k_nearest_neighbors is not a known attribute',
        ]
    
    def write(self, msg):
        if not any(pattern in msg for pattern in self.suppress_patterns):
            self.stream.write(msg)
    
    def flush(self):
        self.stream.flush()

sys.stderr = StderrFilter(sys.__stderr__)

# Microsoft Agent Framework
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient, AzureAISearchContextProvider

# Azure Identity - Use synchronous InteractiveBrowserCredential
# NOTE: DefaultAzureCredential can pick up agent identity context which causes 
# AADSTS82001 errors because agent identities require special token flows.
# InteractiveBrowserCredential authenticates you as a human user via browser device code flow.
from azure.identity import InteractiveBrowserCredential

# Load environment
load_dotenv()

print("✅ Imports loaded successfully")
print("🔇 Noisy warnings suppressed (aiohttp, k_nearest_neighbors)")
print("")
print("⚠️  Authentication Note:")
print("   This notebook uses InteractiveBrowserCredential.")
print("   When you run a query, you'll see a URL and code to authenticate in your browser.")
print("")
print("   Agent identities (Microsoft Entra Agent ID) require special token flows.")
print("   See: https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/microsoft-entra-sdk-for-agent-identities")

✅ Imports loaded successfully
🔇 Noisy warnings suppressed (aiohttp, k_nearest_neighbors)

⚠️  Authentication Note:
   This notebook uses InteractiveBrowserCredential.
   When you run a query, you'll see a URL and code to authenticate in your browser.

   Agent identities (Microsoft Entra Agent ID) require special token flows.
   See: https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/microsoft-entra-sdk-for-agent-identities


In [42]:
# =============================================================================
# Configuration
# =============================================================================

# Azure AI Foundry Project
AI_FOUNDRY_NAME = os.getenv('AI_FOUNDRY_NAME')
AI_PROJECT_NAME = os.getenv('AI_PROJECT_NAME')
PROJECT_ENDPOINT = os.getenv('AI_FOUNDRY_PROJECT_ENDPOINT') or \
    f"https://{AI_FOUNDRY_NAME}.services.ai.azure.com/api/projects/{AI_PROJECT_NAME}"

# Model Deployment
MODEL_DEPLOYMENT = os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME', 'gpt-4o')

# Azure AI Search
SEARCH_ENDPOINT = os.getenv('AZURE_SEARCH_ENDPOINT')
SEARCH_API_KEY = os.getenv('AZURE_SEARCH_API_KEY')

# Azure OpenAI (required for agentic mode)
OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')

# =============================================================================
# Search Indexes - Consolidated Knowledge Bases
# =============================================================================
# These indexes are created by notebook 07 with:
# - Semantic search configuration (required for agentic mode)
# - Vector search with embeddings
# - Real document data from data/index-data/*.jsonl
#
# For simple demos, you can also use agents-us/agents-apac from notebook 05
# (now with semantic config enabled)
SEARCH_INDEXES = {
    'hr': 'hrdocs-index',           # HR policies, company info (50 docs)
    'health': 'healthdocs-index',   # Health benefits, insurance (334 docs)
    # Legacy simple indexes (from notebook 05)
    'us': 'agents-us',
    'apac': 'agents-apac',
}

# Agent definitions directory
AGENTS_DIR = Path('./agents')

print("Configuration")
print("=" * 50)
print(f"Project Endpoint: {PROJECT_ENDPOINT}")
print(f"Model: {MODEL_DEPLOYMENT}")
print(f"Search Endpoint: {SEARCH_ENDPOINT}")
print(f"OpenAI Endpoint: {OPENAI_ENDPOINT}")
print(f"\nSearch Indexes (recommended):")
print(f"   hr:     {SEARCH_INDEXES['hr']} (HR policies, 50 docs)")
print(f"   health: {SEARCH_INDEXES['health']} (Health benefits, 334 docs)")
print(f"\nLegacy indexes (simple demos):")
print(f"   us:     {SEARCH_INDEXES['us']}")
print(f"   apac:   {SEARCH_INDEXES['apac']}")

Configuration
Project Endpoint: https://aifz6vuv4jvgmejw.services.ai.azure.com/api/projects/agent365-project
Model: gpt-4o
Search Endpoint: https://a365-search-6uuruydd4tej6.search.windows.net
OpenAI Endpoint: https://aifz6vuv4jvgmejw.openai.azure.com/

Search Indexes (recommended):
   hr:     hrdocs-index (HR policies, 50 docs)
   health: healthdocs-index (Health benefits, 334 docs)

Legacy indexes (simple demos):
   us:     agents-us
   apac:   agents-apac


In [43]:
# =============================================================================
# Load Agent Configurations from YAML
# =============================================================================

def load_agent_configs(agents_dir: Path) -> dict:
    """Load all agent configurations from YAML files."""
    configs = {}
    
    if not agents_dir.exists():
        print(f"⚠️ Agents directory not found: {agents_dir}")
        return configs
    
    for yaml_file in agents_dir.glob('*.yaml'):
        try:
            with open(yaml_file, 'r') as f:
                config = yaml.safe_load(f)
                agent_name = config.get('name')
                if agent_name:
                    configs[agent_name] = config
                    # Get first index name for display
                    indexes = config.get('tools', {}).get('azure_ai_search', {}).get('indexes', [])
                    index_name = indexes[0].get('name') if indexes else 'none'
                    print(f"  ✅ {agent_name} → {index_name}")
        except Exception as e:
            print(f"  ❌ Error loading {yaml_file.name}: {e}")
    
    return configs

print("Loading agent configurations...")
print("-" * 50)
agent_configs = load_agent_configs(AGENTS_DIR)
print(f"\n📦 Loaded {len(agent_configs)} agent configurations")

Loading agent configurations...
--------------------------------------------------
  ✅ customer-service-agent → agents-us
  ✅ hr-benefits-agent → healthdocs-index
  ✅ fulfillment-agent → agents-us
  ✅ employee-wellness-agent → healthdocs-index
  ✅ regional-apac-agent → agents-apac
  ✅ compliance-agent → agents-us-secure

📦 Loaded 6 agent configurations


In [ ]:
# =============================================================================
# Credential Helper - Import from shared utils.py
# =============================================================================
# Uses InteractiveBrowserCredential with fallback to DeviceCodeCredential.
# Credential is cached to avoid repeated auth prompts.

from utils import get_user_credential, get_graph_token, get_agent_credential

# Get tenant ID from config or environment
TENANT_ID = os.getenv('AZURE_TENANT_ID') or "9249ded8-dff5-4e90-9d80-3ae45c13ec3f"

print("✅ Credential helpers loaded from utils.py")
print(f"   TENANT_ID: {TENANT_ID}")
print("\n   Available functions:")
print("   • get_user_credential() - Interactive browser auth (cached)")
print("   • get_graph_token(config) - Graph API token via MSAL")
print("   • get_agent_credential(tenant, blueprint_id, secret) - Agent identity")


In [ ]:
# =============================================================================
# Identity Diagnostics - Concise Summary
# =============================================================================
import json

def show_identity_info():
    """Display identity configuration summary."""
    
    print("🔐 Identity Configuration")
    print("=" * 50)
    
    # Environment variables
    env_vars = {
        'AZURE_CLIENT_ID': os.getenv('AZURE_CLIENT_ID'),
        'AZURE_TENANT_ID': os.getenv('AZURE_TENANT_ID'),
        'AZURE_CLIENT_SECRET': '***' if os.getenv('AZURE_CLIENT_SECRET') else None,
    }
    
    print("\n📋 Environment Variables:")
    for var, val in env_vars.items():
        status = "✅" if val else "❌"
        display = val[:20] + "..." if val and len(val) > 20 and '***' not in val else val
        print(f"   {status} {var}: {display or 'Not set'}")
    
    # a365 config
    print("\n📋 Agent Blueprint Config:")
    if Path('../a365.generated.config.json').exists():
        with open('../a365.generated.config.json') as f:
            gen = json.load(f)
        print(f"   Blueprint ID: {gen.get('agentBlueprintId', 'N/A')}")
        print(f"   Service Principal: {gen.get('agentBlueprintServicePrincipalObjectId', 'N/A')}")
        print(f"   Secret: {'✅ Set' if gen.get('agentBlueprintClientSecret') else '❌ Not set'}")
    else:
        print("   ⚠️  a365.generated.config.json not found")
    
    # Foundry config
    print("\n📋 Foundry Config:")
    if Path('foundry-config.json').exists():
        with open('foundry-config.json') as f:
            fc = json.load(f)
        foundry = fc.get('foundry', {})
        print(f"   Project: {foundry.get('project_name', 'N/A')}")
        print(f"   Endpoint: {foundry.get('endpoint', 'N/A')[:50]}...")
    else:
        print("   ⚠️  foundry-config.json not found")

show_identity_info()


In [46]:
# =============================================================================
# Verify Current User Identity
# =============================================================================
import jwt  # PyJWT library

async def verify_current_identity():
    """
    Get a token and decode it to see who we're authenticated as.
    This helps verify we're using the right identity for queries.
    """
    print("🔍 Verifying Current Identity...")
    print("-" * 50)
    
    credential = get_user_credential()
    
    try:
        # Get a token for Azure management (common scope)
        token = credential.get_token("https://management.azure.com/.default")
        
        # Decode the token (without verification) to see claims
        # Note: In production, always verify tokens properly
        decoded = jwt.decode(token.token, options={"verify_signature": False})
        
        print(f"✅ Successfully authenticated!")
        print(f"")
        print(f"   👤 User/App Info:")
        print(f"      • Name: {decoded.get('name', 'N/A')}")
        print(f"      • UPN/Email: {decoded.get('upn') or decoded.get('unique_name') or decoded.get('preferred_username', 'N/A')}")
        print(f"      • Object ID (oid): {decoded.get('oid', 'N/A')}")
        print(f"      • App ID (appid/azp): {decoded.get('appid') or decoded.get('azp', 'N/A')}")
        print(f"")
        print(f"   🏢 Tenant Info:")
        print(f"      • Tenant ID: {decoded.get('tid', 'N/A')}")
        print(f"      • Issuer: {decoded.get('iss', 'N/A')}")
        print(f"")
        print(f"   🔐 Token Type:")
        print(f"      • Identity Type (idtyp): {decoded.get('idtyp', 'user (default)')}")
        print(f"      • Subject (sub): {decoded.get('sub', 'N/A')[:20]}...")
        
        # Check for agent-specific claims
        if decoded.get('xms_act_fct') or decoded.get('xms_sub_fct'):
            print(f"")
            print(f"   ⚠️  Agent Identity Claims Detected:")
            print(f"      • Actor Facet: {decoded.get('xms_act_fct', 'N/A')}")
            print(f"      • Subject Facet: {decoded.get('xms_sub_fct', 'N/A')}")
        else:
            print(f"")
            print(f"   ✅ No agent identity claims - this is a regular user token")
            
        return decoded
        
    except Exception as e:
        print(f"❌ Failed to get/decode token: {e}")
        return None

# Run verification
await verify_current_identity()

🔍 Verifying Current Identity...
--------------------------------------------------
⚠️  Detected environment variables that may cause AADSTS82001:
   AZURE_CLIENT_ID = c3ef89f1...
   AZURE_CLIENT_SECRET = phq8Q~QA...
   Using InteractiveBrowserCredential to bypass agent identity context.

✅ Successfully authenticated!

   👤 User/App Info:
      • Name: System Administrator
      • UPN/Email: admin@MngEnvMCAP166727.onmicrosoft.com
      • Object ID (oid): d4215af3-a69c-4640-9004-3cfc07dc032f
      • App ID (appid/azp): 04b07795-8ddb-461a-bbee-02f9e1bf7b46

   🏢 Tenant Info:
      • Tenant ID: 9249ded8-dff5-4e90-9d80-3ae45c13ec3f
      • Issuer: https://sts.windows.net/9249ded8-dff5-4e90-9d80-3ae45c13ec3f/

   🔐 Token Type:
      • Identity Type (idtyp): user
      • Subject (sub): sr55UaRNyfRUje1MYUpQ...

   ⚠️  Agent Identity Claims Detected:
      • Actor Facet: 5 3
      • Subject Facet: 12 3


{'aud': 'https://management.azure.com',
 'iss': 'https://sts.windows.net/9249ded8-dff5-4e90-9d80-3ae45c13ec3f/',
 'iat': 1768867401,
 'nbf': 1768867401,
 'exp': 1768871633,
 'acr': '1',
 'acrs': ['p1', 'urn:user:registersecurityinfo'],
 'aio': 'AYQAe/8bAAAA0z69451lA7FNfrSujVCX2s/saaULGOs3lWn0N829rwH0xtW4LGsprLFMHXlkFW3XeuudUsaGcSAUGqXSxA6oYQSkXXBmqhP2pOVINjVMXr/xPQ7VvNzT8uNRfm6tivktHkPPlfbWzAltxIPgIpLtQrXsXnY1Gqol3WNzGEVFyUc=',
 'amr': ['pwd', 'mfa'],
 'appid': '04b07795-8ddb-461a-bbee-02f9e1bf7b46',
 'appidacr': '0',
 'family_name': 'Administrator',
 'given_name': 'System',
 'groups': ['64a26c50-aed1-40c4-a95f-07c78003bbf0',
  '5778055a-f56b-47ec-bf0f-da9955b28d80',
  '6a8a328d-5943-49b6-9721-9047b8f4a869',
  'b51d76db-c9dd-445e-b1b2-c8a853190df5',
  '6cb68de2-5a1d-4c59-a89e-5f425052c4c9',
  '42a7ccf5-1ffd-4a0d-931d-7315c8b2ddb4',
  'bfe83b5f-240d-46a4-a613-3b5463e518d5',
  '114a8bb8-4515-4ec0-829e-05ac1eee1c7a'],
 'idtyp': 'user',
 'ipaddr': '20.97.10.99',
 'name': 'System Administra

In [ ]:
# =============================================================================
# Validate Entra Agent Identity Association
# =============================================================================
# Verifies agent identity configuration and checks Foundry agent status.

import json
import requests
from pathlib import Path
from azure.ai.projects import AIProjectClient

async def validate_agent_identity_setup():
    """Validate Entra Agent Identity configuration."""
    
    print("🔐 Agent Identity Validation")
    print("=" * 60)
    
    # Load configurations
    a365_config = {}
    gen_config = {}
    
    if Path('../a365.config.json').exists():
        with open('../a365.config.json') as f:
            a365_config = json.load(f)
    
    if Path('../a365.generated.config.json').exists():
        with open('../a365.generated.config.json') as f:
            gen_config = json.load(f)
    
    blueprint_id = gen_config.get('agentBlueprintId')
    sp_id = gen_config.get('agentBlueprintServicePrincipalObjectId')
    has_secret = bool(gen_config.get('agentBlueprintClientSecret'))
    
    # Summary table
    print(f"""
Configuration Status:
  Blueprint ID:      {blueprint_id or '❌ Not configured'}
  Service Principal: {sp_id or '❌ Not configured'}
  Client Secret:     {'✅ Configured' if has_secret else '❌ Not set'}
  Identity Name:     {a365_config.get('agentIdentityDisplayName', 'N/A')}
""")
    
    # Verify service principal in Entra ID
    if sp_id:
        try:
            credential = get_user_credential()
            token = credential.get_token("https://graph.microsoft.com/.default")
            headers = {'Authorization': f'Bearer {token.token}'}
            
            response = requests.get(
                f"https://graph.microsoft.com/v1.0/servicePrincipals/{sp_id}",
                headers=headers
            )
            
            if response.status_code == 200:
                sp = response.json()
                print(f"✅ Service Principal verified: {sp.get('displayName')}")
            else:
                print(f"⚠️  Could not verify SP: {response.status_code}")
        except Exception as e:
            print(f"⚠️  Graph API error: {e}")
    
    # List Foundry agents
    try:
        credential = get_user_credential()
        project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
        agents = list(project_client.agents.list())
        
        if agents:
            print(f"\n📋 Foundry Agents ({len(agents)}):")
            for agent in agents:
                print(f"   • {agent.name} (id: {agent.id})")
        else:
            print("\n📋 No Foundry agents found")
    except Exception as e:
        print(f"⚠️  Could not list agents: {e}")
    
    # Usage guidance
    print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ℹ️  Current Status: Agents use CALLER identity (your token)

To use agent blueprint identity instead:

  from utils import get_agent_credential
  
  agent_cred = get_agent_credential(
      tenant_id="{a365_config.get('tenantId', 'TENANT_ID')}",
      blueprint_id="{blueprint_id or 'BLUEPRINT_ID'}",
      client_secret="<from a365.generated.config.json>"
  )
  project_client = AIProjectClient(endpoint=..., credential=agent_cred)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

await validate_agent_identity_setup()


## Search Connection Verification

Verify the Foundry project has access to our search service.

In [54]:
# =============================================================================
# Setup Tracing for Foundry Agent Operations
# =============================================================================
# OpenTelemetry tracing captures:
# - Agent creation and invocation
# - Tool calls (Azure AI Search queries)
# - Response generation steps
#
# Supports two modes:
# 1. Console tracing (default) - traces printed to notebook output
# 2. Azure Monitor (when APPLICATIONINSIGHTS_CONNECTION_STRING is set in .env)

# Install tracing packages if needed
# !pip install opentelemetry-sdk azure-core-tracing-opentelemetry azure-monitor-opentelemetry

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

# Check for App Insights connection string (set by notebook 05)
APP_INSIGHTS_CONN_STRING = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")

# Configure OpenTelemetry
tracer_provider = TracerProvider()

# Always add console exporter for visibility in notebook
tracer_provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))

# Optionally add Azure Monitor exporter if connection string is available
if APP_INSIGHTS_CONN_STRING:
    try:
        from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter
        azure_exporter = AzureMonitorTraceExporter(connection_string=APP_INSIGHTS_CONN_STRING)
        tracer_provider.add_span_processor(SimpleSpanProcessor(azure_exporter))
        print("✅ Azure Monitor tracing enabled")
        print(f"   Connection: ...{APP_INSIGHTS_CONN_STRING[-30:]}")
    except ImportError:
        print("⚠️  azure-monitor-opentelemetry not installed")
        print("   Run: pip install azure-monitor-opentelemetry")
else:
    print("ℹ️  Azure Monitor tracing not configured")
    print("   Set APPLICATIONINSIGHTS_CONNECTION_STRING in .env (from notebook 05)")

trace.set_tracer_provider(tracer_provider)

# Get tracer for our operations
tracer = trace.get_tracer("foundry-agent-demo")

# Enable content recording to see tool parameters and results
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"

print("\n✅ OpenTelemetry tracing configured")
print("   Traces will be printed to console showing:")
print("   • Agent operations")
print("   • Tool calls (Azure AI Search)")
print("   • Response generation")

ℹ️  Azure Monitor tracing not configured
   Set APPLICATIONINSIGHTS_CONNECTION_STRING in .env (from notebook 05)

✅ OpenTelemetry tracing configured
   Traces will be printed to console showing:
   • Agent operations
   • Tool calls (Azure AI Search)
   • Response generation


In [60]:
# =============================================================================
# Verify Search Connection Setup
# =============================================================================
# Before creating agents, let's verify the Foundry project has access to our search service

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ConnectionType

print("🔍 SEARCH CONNECTION VERIFICATION")
print("=" * 60)
print(f"Your search service: {SEARCH_ENDPOINT}")
print(f"Expected indexes: {list(SEARCH_INDEXES.values())}")
print()

credential = get_user_credential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# List all connections
connections = list(project_client.connections.list())
search_connections = []

print("📡 Foundry project connections:")
for conn in connections:
    conn_type = getattr(conn, 'type', None) or getattr(conn, 'connection_type', 'unknown')
    marker = "🔎" if conn_type == ConnectionType.AZURE_AI_SEARCH else "  "
    print(f"   {marker} {conn.name} ({conn_type})")
    if conn_type == ConnectionType.AZURE_AI_SEARCH:
        search_connections.append(conn)

print()
if search_connections:
    print(f"✅ Found {len(search_connections)} Azure AI Search connection(s)")
    for conn in search_connections:
        print(f"   → {conn.name} (id: {conn.id[:50]}...)")
    
    # Check if our search service is connected
    our_search_name = SEARCH_ENDPOINT.replace("https://", "").replace(".search.windows.net", "")
    matched = any(our_search_name.lower() in c.name.lower() for c in search_connections)
    
    if matched:
        print(f"\n✅ Your search service '{our_search_name}' appears to be connected")
    else:
        print(f"\n⚠️  Your search service '{our_search_name}' is NOT connected!")
        print(f"   The connected search service(s): {[c.name for c in search_connections]}")
        print()
        print("   To fix this, add your search service in AI Foundry portal:")
        print("   1. Go to AI Foundry → Your Project → Management → Connected Resources")
        print("   2. Click '+ New Connection'")
        print("   3. Select 'Azure AI Search'")
        print(f"   4. Connect to: {SEARCH_ENDPOINT}")
else:
    print("❌ No Azure AI Search connections found in project")
    print("   Add one in AI Foundry portal → Management → Connected Resources")

🔍 SEARCH CONNECTION VERIFICATION
Your search service: https://a365-search-6uuruydd4tej6.search.windows.net
Expected indexes: ['hrdocs-index', 'healthdocs-index', 'agents-us', 'agents-apac']

📡 Foundry project connections:
   🔎 lab511searchl4nj2d6mf7eejc7 (ConnectionType.AZURE_AI_SEARCH)
   🔎 a365-search-connection (ConnectionType.AZURE_AI_SEARCH)
      kb-hr-benefits-assista-eejc7 (ConnectionType.REMOTE_TOOL)
      agenticdevops (ConnectionType.APPLICATION_INSIGHTS)

✅ Found 2 Azure AI Search connection(s)
   → lab511searchl4nj2d6mf7eejc7 (id: /subscriptions/63862159-43c8-47f7-9f6f-6c63d56b0e1...)
   → a365-search-connection (id: /subscriptions/63862159-43c8-47f7-9f6f-6c63d56b0e1...)

⚠️  Your search service 'a365-search-6uuruydd4tej6' is NOT connected!
   The connected search service(s): ['lab511searchl4nj2d6mf7eejc7', 'a365-search-connection']

   To fix this, add your search service in AI Foundry portal:
   1. Go to AI Foundry → Your Project → Management → Connected Resources
   2. 